# 09 · Navigation filter calibration sweep (Kaggle)

The filter is over-confident: NEES at the decision averaged 8-14 where a consistent 3-D filter gives 3. This flies the same 100 landings (seeds 0-99, harder aim points) under five settings of the correlated-error inflation (`corr_factor`) and the process noise (`sigma_a`), and reports NEES, 95% coverage and the decision metrics for each. No detector is used (it does not affect navigation), so each landing is fast.

Settings: **GPU T4 x2**, **Internet on**. No dataset input is needed. Save Version, Save & Run All (Commit); about 3-4 hours.


In [ ]:
!git clone -q https://github.com/aryanyaksh-art/touch_down.git /tmp/touch_down || (cd /tmp/touch_down && git pull -q)
!sh /tmp/touch_down/scripts/kaggle_setup.sh 2>&1 | tail -5

In [ ]:
import os, subprocess
CONFIGS = {
    'baseline (corr 1.5, sigma_a 2e-6)': dict(corr=1.5, sa=2e-6),
    'corr 2.0': dict(corr=2.0, sa=2e-6),
    'corr 2.5': dict(corr=2.5, sa=2e-6),
    'corr 3.0': dict(corr=3.0, sa=2e-6),
    'corr 2.5 + sigma_a 1e-5': dict(corr=2.5, sa=1e-5),
}
ENV = dict(os.environ, TOUCHDOWN_DATA='/tmp/data', TOUCHDOWN_BLENDER='/tmp/blender/blender')
SEEDS = (0, 100)
dirs = {}
for name, c in CONFIGS.items():
    d = '/kaggle/working/sweep/' + name.split(' (')[0].replace(' ', '_').replace('+', 'and')
    dirs[name] = d
    cmd = ['python', '-m', 'touchdown.sim.montecarlo', '--out', d, '--seeds', str(SEEDS[0]), str(SEEDS[1]), '--device', 'GPU',
           '--aim-mode', 'random_safe', '--corr-factor', str(c['corr']), '--sigma-a', str(c['sa'])]
    print('==', name, flush=True)
    subprocess.run(cmd, cwd='/tmp/touch_down', env=ENV, stdout=subprocess.DEVNULL, check=True)
    print('   done,', len(os.listdir(d)), 'files', flush=True)

In [ ]:
import sys
sys.path.insert(0, '/tmp/touch_down')
from touchdown.analysis.calibration import load, summarize, markdown
table = markdown({k: summarize(load(d)) for k, d in dirs.items()})
print(table)
open('/kaggle/working/calibration.md', 'w').write(table + '\n')